# Airline Flight Data Analysis Using Python

**Goal:** analyse real U.S. airline on-time performance data to understand flight delays, airline punctuality, departure-time patterns and airport performance, using exploratory data analysis (EDA).

**How this notebook works**
- The heavy lifting lives in `src/data_analysis.py` as small, documented functions. This notebook calls them one step at a time and explains each step, so you can read the notebook first and open the matching function when you want the details.
- **Nothing in this notebook is hardcoded.** All numbers, tables, charts and findings are computed from the CSV files you download (see `data/README.md`).
- Run it top to bottom with **Kernel -> Restart & Run All**. The notebook is saved *without* outputs because the data is not included in the repository.

**Sections:** 1 Setup - 2 Definitions - 3 Load & inspect - 4 Clean - 5 Derived columns - 6 EDA (7 questions) - 7 Findings & recommendations - 8 Save outputs - 9 Limitations

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

# Make the notebook find src/data_analysis.py (works whether you open it from the notebooks/ folder or the project root)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

import data_analysis as da

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
print("Project folder:", PROJECT_ROOT)

## 2. Dataset and definitions

**Dataset:** U.S. Bureau of Transportation Statistics (BTS), *Reporting Carrier On-Time Performance* - one row per scheduled domestic flight by U.S. airlines. Source, usage conditions, download steps and the full column list are in `data/README.md`.

**Key delay fields**

| Field | Meaning |
|---|---|
| `CRSDepTime` | Scheduled departure time (hhmm) |
| `DepDelay` | Minutes departed after schedule. **Negative = left early.** Empty if cancelled |
| `DepDel15` | BTS flag: 1 if `DepDelay >= 15` |
| `Cancelled` | 1 if the flight was cancelled |

**Definitions used throughout**
- **Delayed flight** = departed **15+ minutes** after schedule (`DepDelay >= 15`), the same rule as BTS's `DepDel15`.
- **Cancelled flights** never departed, so they have no delay value. They are **excluded** from delay percentages and averages and analysed separately.
- **Flights with a missing delay value that were not cancelled** are also excluded from delay statistics (we do not guess), but are reported.
- **Average delay** = mean `DepDelay` in minutes (early departures are negative). We also show the **median**, because a few very long delays can inflate the mean.
- We analyse **departure** delay. BTS's own headline "on-time" statistic uses *arrival* delay, so our percentages will differ from it.

## 3. Load and inspect the data

In [ ]:
df_raw, files = da.load_data()          # reads every CSV inside the data/ folder
print("Files loaded:", [f.name for f in files])
print("Rows, columns:", df_raw.shape)
df_raw.head()

In [ ]:
# Data types of each column (we convert them in the cleaning step)
df_raw.dtypes

In [ ]:
# Missing values per column. Cancelled flights are expected to have empty DepDelay / DepTime.
da.missing_value_report(df_raw)

In [ ]:
# Quick numeric summary of the raw delay column - look for anything strange
df_raw["DepDelay"].describe()

## 4. Data cleaning

`da.clean_data()` converts data types, looks for duplicates, missing and inconsistent values, and checks for unusual delay values. It returns the cleaned table **and a log** explaining what it found and what it did.

The guiding rule: **do not delete flights without a reason.** Cancelled flights, missing delays and very long delays are kept in the table and handled with filters, so nothing is silently lost.

In [ ]:
df, cleaning_log = da.clean_data(df_raw)
pd.set_option("display.max_colwidth", 140)
cleaning_log

**Reading the log:** only exact duplicate rows (and rows with unreadable dates, if any) are removed. Everything else is kept and flagged. Extreme delays are kept because overnight delays do happen; the histogram clips its axis only for readability.

## 5. Derived columns (feature engineering)

`da.add_features()` creates columns that make the analysis easy:

| New column | What it is |
|---|---|
| `dep_delay_known` | True when the flight departed and has a delay value |
| `is_delayed` | True when `DepDelay >= 15` |
| `flight_status` | On-time / Delayed / Cancelled / Unknown |
| `sched_hour` | Scheduled departure hour, 0-23 (from `CRSDepTime`) |
| `dep_period` | Night (00-05), Morning (06-11), Afternoon (12-17), Evening (18-23) |
| `day_name`, `year_month` | Day of week and month of the flight |
| `airline_name` | Readable airline name from the code |

We group by the **scheduled** hour because that is the time passengers book and airlines plan around.

In [ ]:
df = da.add_features(df)
df[["FlightDate", "Reporting_Airline", "airline_name", "Origin", "CRSDepTime", "sched_hour", "dep_period",
    "DepDelay", "is_delayed", "flight_status"]].head(8)

## 6. Exploratory data analysis

We first run every analysis once with `da.compute_all()`, then walk through the seven questions. Each table is computed from your data.

In [ ]:
results = da.compute_all(df)

# Columns we will show in most tables
cols = ["scheduled_flights", "flights_departed", "avg_dep_delay", "median_dep_delay", "pct_delayed", "pct_cancelled"]
print("Analysis ready. Tables available:", [k for k in results if k not in ("metrics", "min_airline", "min_airport")])

### Q1 and Q2. What percentage of flights were delayed, and what was the average departure delay?

In [ ]:
# The calculation done step by step with plain pandas
departed = df[df["dep_delay_known"]]            # only flights that actually departed with a recorded delay
n_departed = len(departed)
n_delayed = int(departed["is_delayed"].sum())

print(f"Flights with a recorded departure delay: {n_departed:,}")
print(f"Delayed (15+ min late):                 {n_delayed:,}  ({n_delayed / n_departed:.1%})")
print(f"Average departure delay (mean):         {departed['DepDelay'].mean():.1f} min")
print(f"Median departure delay:                 {departed['DepDelay'].median():.1f} min")
print(f"Left early (negative delay):            {(departed['DepDelay'] < 0).mean():.1%}")

In [ ]:
# All headline numbers in one table
pd.Series(results["metrics"]).to_frame("value")

#### Chart 1 - Proportion of delayed, on-time and cancelled flights

In [ ]:
display(results["status"])
da.plot_status_share(results["status"], save_dir=da.CHART_DIR)
plt.show()

#### Chart 2 - Distribution of departure delays
Most flights cluster near zero and the distribution has a long right tail (a few very long delays). That is why we also report the median.

In [ ]:
da.plot_delay_histogram(df, save_dir=da.CHART_DIR)
plt.show()

### Q3. Which airlines had the highest and lowest average delays?
Only airlines with a minimum number of flights are ranked, so tiny carriers do not distort the ranking.

In [ ]:
print(f"Airlines with at least {results['min_airline']:,} scheduled flights, sorted from highest to lowest average delay:")
results["airline"][["label"] + cols]

#### Chart 3 - Average departure delay by airline

In [ ]:
da.plot_airline_delay(results["airline"], results["min_airline"], save_dir=da.CHART_DIR)
plt.show()

### Q4. Which departure hours or days had more delays?

In [ ]:
results["hour"][cols]

#### Chart 4 - Delays by scheduled departure hour

In [ ]:
da.plot_by_hour(results["hour"], save_dir=da.CHART_DIR)
plt.show()

In [ ]:
# By broad time of day, then by day of week
display(results["period"][cols])
results["dow"][cols]

#### Chart 5 - Delays by day of week, and Chart 6 - day x hour heatmap

In [ ]:
da.plot_by_day(results["dow"], save_dir=da.CHART_DIR)
plt.show()

In [ ]:
da.plot_heatmap(results["heatmap"], save_dir=da.CHART_DIR)
plt.show()

### Q5. How did delays vary over time and by month?

In [ ]:
results["month"][cols]

#### Chart 7 - Delay trend over time (daily average with a 7-day rolling average)

In [ ]:
da.plot_daily_trend(results["daily"], save_dir=da.CHART_DIR)
plt.show()

#### Chart 8 - Delay by month
Shown only when at least two months of data are loaded.

In [ ]:
fig = da.plot_monthly(results["month"], save_dir=da.CHART_DIR)
if fig is None:
    print("Only one month of data is loaded, so the by-month chart is skipped. Download more months to enable it.")
else:
    plt.show()

### Q6. Which airports experienced higher average delays?
These are **origin** airports (where the flight departed from), restricted to airports with a minimum number of flights.

In [ ]:
print(f"Airports with at least {results['min_airport']:,} scheduled flights.")
print("Highest average delay:")
display(results["airport"][cols].head(10))
print("Lowest average delay:")
results["airport"][cols].tail(10).sort_values("avg_dep_delay")

#### Chart 9 - Airport performance

In [ ]:
da.plot_airports(results["airport"], results["min_airport"], save_dir=da.CHART_DIR)
plt.show()

### Q7. What patterns can be observed in cancellations?
Cancelled flights have no delay, so this is a separate look at reliability: cancellation rate per airline and the reason codes BTS records.

In [ ]:
print("Reasons for cancellation (BTS reason codes):")
display(results["cancel_reasons"])
print("Airlines with the highest cancellation rate:")
results["airline"].sort_values("pct_cancelled", ascending=False)[["label", "scheduled_flights", "cancelled", "pct_cancelled"]].head(10)

#### Chart 10 - Cancellations

In [ ]:
fig = da.plot_cancellations(results["airline"], results["cancel_reasons"], save_dir=da.CHART_DIR)
if fig is None:
    print("No cancelled flights in this sample.")
else:
    plt.show()

## 7. Findings and recommendations

The text below is **generated from the results above** by `da.generate_findings()` and `da.generate_recommendations()`. If you change the months you download, the findings change automatically. Each finding states the evidence (numbers) and what it does and does not mean.

> **Reminder:** these are patterns (correlations) in the months you downloaded. They do not prove *why* delays happen.

In [ ]:
findings = da.generate_findings(df, results)
recommendations = da.generate_recommendations(df, results)
display(Markdown(da.render_findings_markdown(findings, recommendations)))

## 8. Save the outputs
This writes a full text report and CSV tables to `outputs/reports/` and makes sure every chart is saved in `outputs/charts/`.

In [ ]:
da.save_tables(results)
report_path = da.write_report(da.REPORT_DIR / "analysis_report.md", files, cleaning_log, results, findings, recommendations)
charts = da.make_all_charts(df, results)
print("Report:", report_path)
print("Charts saved:", len(charts))
for c in charts:
    print("  ", c.name)

## 9. Limitations and next steps

**Limitations**
- Only the months you downloaded are analysed; delays are strongly seasonal, so results do not describe a whole year.
- Averages are not adjusted for route, airport mix, aircraft type or weather. A difference between two airlines is not proof that one is better run.
- We analyse **departure** delays; BTS's on-time statistics are based on **arrival** delays.
- Airline codes are mapped to names with a small convenience table; check any code you are unsure about.

**Why no SQL or machine learning?** The data is one flat table, and Pandas `groupby` answers every question clearly, so SQL would add complexity without extra value. The goal is to *describe* delay patterns, not to predict them, so a model is not justified here. A sensible extension would be predicting whether a flight will be delayed, but only after a baseline and proper train/test split.

**Next steps**
- Add more months or a full year and re-run (nothing else needs to change).
- Use the BTS delay-cause columns (`CarrierDelay`, `WeatherDelay`, `NASDelay`, `LateAircraftDelay`) for a root-cause analysis.
- Compare delays by route and distance; build a Power BI / Tableau dashboard from `outputs/reports/table_*.csv`.